# Memory AI Lab — 02b : Transformer Boundary Detector

**GPU requis** → `Exécution > Modifier le type d'exécution > GPU T4`

## Pourquoi un Transformer vs TCN ?

```
TCN  : champ réceptif ~42 msgs (3 blocs × dilation 1-8 × kernel 3)
TFM  : self-attention globale → champ = fenêtre entière (100 msgs)
```

Pour les ruptures de sujet longue portée, le Transformer voit plus loin.

## Architecture v3 (projections séparées)

```
mE5 embeddings [T × 768]               scalar features [T × 3]
        ↓ Linear(768 → 128)                    ↓ Linear(3 → 32) → GELU → Linear(32 → 128)
        └──────────────── addition ────────────────┘
                    [T × 128]   h = emb_proj + scalar_proj
                        ↓ Transformer Encoder × 2 (4 heads, FFN 512d, pre-LN)
                        ↓ Linear(128 → 32) → GELU → Linear(32 → 1) → sigmoid
                   P(frontière) [T × 1]
Total : ~502K paramètres
```

**Features scalaires (issues de l'éval qualitative Claude) :**
- `gap_log`        : log1p(gap_min) / log1p(1440) — rupture temporelle
- `msg_length_log` : log1p(len) / log1p(500)      — verbosité (débat=long, médias=court)
- `is_media`       : 1 si URL ou < 15 chars        — partage passif → réduit FP

**Pourquoi projections séparées vs joint Linear(771→128) ?**
Dans une projection jointe, les 3 scalaires représentent < 0.5% du gradient
(768 dims d'embedding écrasent le signal). Le MLP dédié leur donne leur propre espace.

## Données requises (`memory_ai_data/`)
```
group_anon.txt
group_gold_tune.json
group_gold_test.json
group_embeddings_me5.npy   (cache optionnel)
```

Sortie : `boundary_detector_tfm.pt` (ne touche pas `boundary_detector_tcn.pt`)

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO     = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
print('✓ OK')
print()
print('⚠️  Si première exécution : Exécution > Redémarrer la session,')
print('   puis relancer à partir de la cellule 3.')

In [ ]:
# ── CELLULE 3 : Google Drive + Copie locale ────────────────────────────────
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/memory_ai_data'
LOCAL_DIR = '/content/data'
os.makedirs(LOCAL_DIR, exist_ok=True)

FILES_TO_COPY = [
    'group_anon.txt',
    'group_gold_tune.json',
    'group_gold_test.json',
    'group_embeddings_me5.npy',
]
for fname in FILES_TO_COPY:
    src, dst = f'{DRIVE_DIR}/{fname}', f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        print(f'  Copie {fname} ...', end=' ', flush=True)
        shutil.copy2(src, dst)
        print('✓')
    elif os.path.exists(dst):
        print(f'  {fname} déjà en local ✓')
    else:
        print(f'  {fname} absent sur Drive (ignoré)')

DATA_DIR = LOCAL_DIR
print(f'\n✓ DATA_DIR = {DATA_DIR}')

In [ ]:
# ── CELLULE 4 : Parse + Embeddings mE5-base ───────────────────────────────
import numpy as np
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from parsers.whatsapp_parser import parse_whatsapp_chat

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

MODEL_NAME  = 'intfloat/multilingual-e5-base'
PREFIX      = 'passage: '
EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings_me5.npy'

all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
texts = [PREFIX + a.content for a in all_artifacts]
print(f'[1/2] {len(texts)} messages parsés')

DRIVE_CACHE = f'{DRIVE_DIR}/group_embeddings_me5.npy'
if not EMBED_CACHE.exists() and os.path.exists(DRIVE_CACHE):
    shutil.copy2(DRIVE_CACHE, EMBED_CACHE)

if EMBED_CACHE.exists():
    all_embeddings = np.load(EMBED_CACHE)
    assert len(all_embeddings) == len(texts), 'Cache périmé — supprimer et relancer'
    print('[2/2] Embeddings chargés depuis cache')
else:
    print(f'[2/2] Calcul sur {device} (mE5-base 768d) ...')
    model = SentenceTransformer(MODEL_NAME, device=device)
    all_embeddings = model.encode(
        texts, batch_size=256, show_progress_bar=True,
        device=device, convert_to_numpy=True
    ).astype(np.float32)
    np.save(EMBED_CACHE, all_embeddings)
    shutil.copy2(EMBED_CACHE, DRIVE_CACHE)
    print(f'  Sauvegardé → {EMBED_CACHE}')

print(f'  Shape : {all_embeddings.shape}')

In [ ]:
# ── CELLULE 5 : Split tune_early (Protocole B) ────────────────────────────
# Identique au notebook TCN — boundary detector entraîné sur tune_early UNIQUEMENT.
# tune_late reste réservé à Optuna dans 01_eval_ari.ipynb.
import json

TUNE_SPLIT = 0.65   # doit être identique dans 01_eval_ari.ipynb

def load_split(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    n = len(data['artifacts'])
    y_true = [None] * n
    for ep in data['episodes']:
        for idx in range(ep['start_idx'], ep['end_idx'] + 1):
            if idx < n:
                y_true[idx] = ep['episode_id']
    return data['artifacts'], y_true, data['episodes'], data['meta']

tune_arts_all, y_true_tune_all, tune_eps_all, tune_meta = load_split(f'{DATA_DIR}/group_gold_tune.json')
test_arts,     y_true_test,     test_eps,     test_meta  = load_split(f'{DATA_DIR}/group_gold_test.json')

n_tune  = len(tune_arts_all)
n_split = int(n_tune * TUNE_SPLIT)

arts_early = all_artifacts[:n_split]
emb_early  = all_embeddings[:n_split]
y_early    = y_true_tune_all[:n_split]

arts_test  = all_artifacts[n_tune:n_tune + len(test_arts)]
emb_test   = all_embeddings[n_tune:n_tune + len(test_arts)]

n_early_eps = len(set(y for y in y_early if y is not None))

print(f'Tune total  : {n_tune} msgs · {len(tune_eps_all)} épisodes · {tune_meta["period"]}')
print(f'Tune early  : {n_split} msgs · ~{n_early_eps} épisodes  → Transformer')
print(f'Tune late   : {n_tune - n_split} msgs                   → Optuna (01_eval_ari.ipynb)')
print(f'Test        : {len(test_arts)} msgs · {len(test_eps)} épisodes · {test_meta["period"]}')

In [ ]:
# ── CELLULE 6 : Labels de frontière ───────────────────────────────────────
from boundary_detector_tcn import extract_boundary_labels

y_early_bin = extract_boundary_labels(y_early, len(arts_early))
y_test_bin  = extract_boundary_labels(y_true_test, len(arts_test))

n_pos = int(y_early_bin.sum())
n_neg = len(y_early_bin) - n_pos

print(f'Labels tune_early : {n_pos} frontières · {n_neg} continuations')
print(f'Ratio             : 1:{n_neg // max(n_pos,1)} → pos_weight≈{n_neg // max(n_pos,1)}')
print(f'Labels test       : {int(y_test_bin.sum())} frontières / {len(y_test_bin)} msgs')

In [ ]:
# ── CELLULE 7 : Entraînement TransformerBoundaryDetector ──────────────────
import importlib, sys
for _mod in list(sys.modules.keys()):
    if 'boundary_detector' in _mod:
        del sys.modules[_mod]

from boundary_detector_transformer import TransformerBoundaryDetector

# ── Hyperparamètres ───────────────────────────────────────────────────────
# d_model=128, n_heads=4, n_layers=2 : ~500K params (stable sur ~5K msgs)
# lr=5e-4 : plus faible que TCN (1e-3) — transformers plus sensibles en début
# warmup : 5% des epochs pour stabiliser l'attention (codé dans fit_sequence)

detector = TransformerBoundaryDetector(
    device      = device,
    d_model     = 128,
    n_heads     = 4,
    n_layers    = 2,
    dropout     = 0.15,
    window_size = 100,
)

detector.fit_sequence(
    embeddings   = emb_early,
    artifacts    = arts_early,
    y_true       = y_early,
    n_epochs     = 60,
    lr           = 5e-4,
    weight_decay = 1e-4,
    batch_size   = 32,
    focal_gamma  = 2.0,
    stride_min   = 10,
    stride_max   = 25,
    val_split    = 0.10,
    verbose      = True,
)

print('\n✓ Entraînement terminé')

In [ ]:
# ── CELLULE 8 : Optimisation seuil sur tune_early ─────────────────────────
# Même cible que TCN : recall ≥ 0.85 (Stage 2 compense les faux positifs)
MIN_RECALL = 0.85

detector.optimize_threshold_sequence(
    embeddings = emb_early,
    artifacts  = arts_early,
    y_true     = y_early,
    min_recall = MIN_RECALL,
)

print(f'\nSeuil retenu : {detector.threshold:.3f}')

In [ ]:
# ── CELLULE 9 : Évaluation sur TEST — une seule fois ──────────────────────
import numpy as np

probs_test = detector.predict_proba_sequence(emb_test, arts_test)
preds_test = (probs_test >= detector.threshold).astype(int)

tp   = int(((preds_test == 1) & (y_test_bin == 1)).sum())
fp   = int(((preds_test == 1) & (y_test_bin == 0)).sum())
fn   = int(((preds_test == 0) & (y_test_bin == 1)).sum())
prec = tp / (tp + fp + 1e-8)
rec  = tp / (tp + fn + 1e-8)
f1   = 2 * prec * rec / (prec + rec + 1e-8)
n_b  = int(preds_test.sum())
n_g  = int(y_test_bin.sum())

# Comparaison avec TCN
print(f"""
╔══ TRANSFORMER BOUNDARY DETECTOR — Résultat TEST ══╗
║  Précision  : {prec:.4f}   (TCN baseline ≈ 0.55)  ║
║  Rappel     : {rec:.4f}                            ║
║  F1         : {f1:.4f}                             ║
╠═══════════════════════════════════════════════════╣
║  Frontières : {n_b} prédit / {n_g} gold            ║
║  Seuil      : {detector.threshold:.3f}             ║
╚═══════════════════════════════════════════════════╝

→ Précision > 0.60 : Transformer améliore le TCN ✓
→ Précision ≈ 0.55 : parité — tester n_layers=3 ou d_model=256
→ Précision < 0.55 : relancer l'entraînement (variance init aléatoire)
""")

In [ ]:
# ── CELLULE 10 : Sauvegarde → Drive ───────────────────────────────────────
# Sauvegardé sous boundary_detector_tfm.pt — ne touche PAS au TCN.
# 01_eval_ari.ipynb le chargera automatiquement en priorité.
import shutil

LOCAL_PATH = f'{DATA_DIR}/boundary_detector_tfm.pt'
DRIVE_PATH = f'{DRIVE_DIR}/boundary_detector_tfm.pt'

detector.save(LOCAL_PATH)
shutil.copy2(LOCAL_PATH, DRIVE_PATH)

print(f'✓ Transformer detector sauvegardé → Drive')
print(f'  Seuil : {detector.threshold:.3f}')
print()
print('Étape suivante : ouvrir 01_eval_ari.ipynb')
print('  Le notebook chargera boundary_detector_tfm.pt en priorité.')

## Variantes à tester si F1 < 0.60

```python
# Option A — Plus de profondeur
detector = TransformerBoundaryDetector(n_layers=3, d_model=128, n_heads=4)

# Option B — Plus de capacité
detector = TransformerBoundaryDetector(n_layers=2, d_model=256, n_heads=8)

# Option C — LR plus faible (convergence plus stable)
detector.fit_sequence(..., lr=2e-4, n_epochs=80)
```

## Interprétation des résultats

| Précision TEST | Interprétation | Action |
|---|---|---|
| > 0.65 | Net gain vs TCN | Lancer Optuna → ARI attendu > 0.90 |
| 0.58-0.65 | Gain modéré | Lancer Optuna, probablement ARI ~0.90 |
| 0.50-0.58 | Parité TCN | Relancer 2-3 fois, garder meilleur val_F1 |
| < 0.50 | Régression | Revoir hyperparamètres (lr, n_layers) |